# CS431 — Paddle Phase 1: ICDAR2015 external checkpoint validation
Notebook do repo CS431 tổ chức, dùng evaluator upstream PaddleOCR đã pin. Không phải notebook của tác giả paper.
Chạy đủ test set, xem metric/overlays rồi ghi đánh giá. Exact official Paddle 83,8% reproduction: NOT VERIFIED.


In [ ]:
import sys, subprocess, json
from pathlib import Path
BASE = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/content')
REPO = BASE / 'CS431'
if not (REPO / '.git').is_dir():
    subprocess.run(['git','clone','https://github.com/ncat218/CS431.git',str(REPO)], check=True)
print('Repo:', REPO)
print((REPO/'docs/PADDLE.md').read_text())
subprocess.run([sys.executable, str(REPO/'run.py'), 'paddle', 'status'], cwd=REPO, check=False)


## Môi trường tách riêng
Python 3.11/3.12 Linux. CUDA wheel mặc định cu126; chọn cu118 **trước Phase 1** nếu driver yêu cầu.
Setup không cài driver hoặc sửa môi trường Monkey. Nếu setup/probe thất bại, dừng và đọc logs.


In [ ]:
CUDA_WHEEL = 'cu126'
VENV = BASE / 'cs431-paddle'
subprocess.run([sys.executable, str(REPO/'paddle/tools/setup_paddle.py'), '--venv', str(VENV), '--cuda', CUDA_WHEEL], cwd=REPO, check=True)
PYTHON = str(VENV/'bin/python')
subprocess.run([PYTHON, str(REPO/'paddle/tools/prepare_paddle_model.py')], cwd=REPO, check=True)
GPUS = '0,1' if Path('/kaggle/working').exists() else '0'
print('GPU IDs:', GPUS, '(T4x2: 0,1; one A100: 0)')


## Phase 1 — khai báo dữ liệu thật
Cần đủ img_1 … img_500 và gt_img_1.txt … gt_img_500.txt. Không dùng train split.
Liệt kê thư mục trước, rồi điền đường dẫn và nguồn ICDAR2015 đã attach. Không suy đoán đường dẫn.


In [ ]:
INPUT_ROOT = Path('/kaggle/input') if Path('/kaggle/input').exists() else Path('/content')
print('Available input directories:')
for item in sorted(INPUT_ROOT.iterdir()):
    if item.is_dir(): print(item)
ICDAR_IMAGES = ''  # Thư mục chứa 500 test images (có thể tìm đệ quy bên trong)
ICDAR_GT = ''      # Thư mục chứa 500 localization/transcription GT .txt
DATASET_SOURCE = ''  # URL/phiên bản/nguồn dữ liệu thực tế, để reviewer đối chiếu
if not ICDAR_IMAGES or not ICDAR_GT or not DATASET_SOURCE:
    raise ValueError('Điền ba trường ICDAR_IMAGES, ICDAR_GT và DATASET_SOURCE trước khi chạy Phase 1.')
subprocess.run([PYTHON, str(REPO/'run.py'), 'paddle','phase1',
    '--images',ICDAR_IMAGES,'--gt',ICDAR_GT,'--dataset-source',DATASET_SOURCE,'--gpus',GPUS], cwd=REPO, check=True)


## Xem kết quả trước khi quyết định
Xem số liệu, provenance, per-image counts, logs và đủ 20 overlays. Xanh=GT valid,
xám=GT ignored, đỏ=prediction. Nếu ảnh/GT/config sai hoặc kết quả bất thường, sửa và chạy lại;
không ghi nhận chấp nhận để vượt gate. Không có ngưỡng so với 83,8% vì đây là benchmark khác.


In [ ]:
from IPython.display import display, Markdown
from PIL import Image
PHASE1 = REPO/'paddle/phase1/report'
display(Markdown((PHASE1/'REVIEW_ME.md').read_text()))
print(json.dumps(json.loads((PHASE1/'metrics.json').read_text()),indent=2))
print('Inventory:',PHASE1/'dataset_inventory.json')
print('Worker logs:',PHASE1/'inference')
for image in sorted((PHASE1/'overlays').glob('*.jpg')):
    print(image.name)
    with Image.open(image) as src:
        preview=src.copy(); preview.thumbnail((1100,800)); display(preview)


## Người thực hiện ghi nhận đánh giá Phase 1
Chỉ điền sau khi đã kiểm tra các kết quả phía trên. Đánh giá phải nêu P/R/Hmean,
nhận xét overlays và giới hạn/provenance. Cell này mặc định dừng để không tự chuyển Phase 2.
Tên và ghi chú là quyết định của người thực hiện; không phải một bước tự approve của trợ lý.


In [ ]:
REVIEWER = ''
REVIEW_NOTES = ''
if not REVIEWER.strip() or len(REVIEW_NOTES.strip()) < 20:
    raise ValueError('Cần xem kết quả rồi điền REVIEWER và nhận định thực tế để mở gate Phase 2.')
subprocess.run([PYTHON,str(REPO/'run.py'),'paddle','review','--reviewer',REVIEWER,'--notes',REVIEW_NOTES], cwd=REPO, check=True)
